'''Objective: Predict heart disease risk from CDC BRFSS 2022 survey data to enable early 
intervention.
Dataset: 445,132 records, 39 features, binary target (HadHeartAttack).
Challenge: TBD after EDA—will assess class distribution, missing values, feature correlations.
Success Metric:TBD after model exploration—will determine based on imbalance severity 
and business context.'''


In [2]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# This ensures all 40 columns are displayed in the output
pd.set_option('display.max_columns', None)

# Loading the dataset
data = pd.read_csv('Heart_disease.csv')
data.info()
# Display the first five rows of the dataset
print(data.head()) 


<class 'pandas.DataFrame'>
RangeIndex: 445132 entries, 0 to 445131
Data columns (total 40 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   State                      445132 non-null  str    
 1   Sex                        445132 non-null  str    
 2   GeneralHealth              443934 non-null  str    
 3   PhysicalHealthDays         434205 non-null  float64
 4   MentalHealthDays           436065 non-null  float64
 5   LastCheckupTime            436824 non-null  str    
 6   PhysicalActivities         444039 non-null  str    
 7   SleepHours                 439679 non-null  float64
 8   RemovedTeeth               433772 non-null  str    
 9   HadHeartAttack             442067 non-null  str    
 10  HadAngina                  440727 non-null  str    
 11  HadStroke                  443575 non-null  str    
 12  HadAsthma                  443359 non-null  str    
 13  HadSkinCancer              441989 non-nu

In [3]:
# Data cleaning 
data.describe(include = 'all')


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,HadAngina,HadStroke,HadAsthma,HadSkinCancer,HadCOPD,HadDepressiveDisorder,HadKidneyDisease,HadArthritis,HadDiabetes,DeafOrHardOfHearing,BlindOrVisionDifficulty,DifficultyConcentrating,DifficultyWalking,DifficultyDressingBathing,DifficultyErrands,SmokerStatus,ECigaretteUsage,ChestScan,RaceEthnicityCategory,AgeCategory,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
count,445132,445132,443934,434205.000000,436065.000000,436824,444039,439679.000000,433772,442067,440727,443575,443359,441989,442913,442320,443206,442499,444045,424485,423568,420892,421120,421217,419476,409670,409472,389086,431075,436053,416480.000000,403054.000000,396326.000000,398558,379005,398011,368092,362616,394509,394368
unique,54,2,5,NaN,NaN,4,2,NaN,4,2,2,2,2,2,2,2,2,2,4,2,2,2,2,2,2,4,4,2,5,13,NaN,NaN,NaN,2,2,2,2,4,2,3
top,Washington,Female,Very good,NaN,NaN,Within past year (anytime less than 12 months ...,Yes,NaN,None of them,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Never smoked,Never used e-cigarettes in my entire life,No,"White only, Non-Hispanic",Age 65 to 69,NaN,NaN,NaN,Yes,No,Yes,No,"No, did not receive any tetanus shot in the pa...",No,No
freq,26152,235893,148444,NaN,NaN,350944,337559,NaN,233455,416959,414176,424336,376665,406504,407257,350910,422891,291351,368722,385539,399910,370792,353039,404404,387029,245955,311988,223221,320421,47099,NaN,NaN,NaN,210891,249919,209256,215604,121493,377324,270055
mean,NaN,NaN,NaN,4.347919,4.382649,NaN,NaN,7.022983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.702691,83.074470,28.529842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,8.688912,8.387475,NaN,NaN,1.502425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.107177,21.448173,6.554889,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.910000,22.680000,12.020000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,6.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.630000,68.040000,24.130000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,7.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.700000,80.740000,27.440000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,3.000000,5.000000,NaN,NaN,8.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.780000,95.250000,31.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Check for duplicates
print('Total Number of duplicated rows:',data.duplicated().sum())

# Show all the duplicated rows 
all_duplicates = data[data.duplicated(keep = False)] # Return all the duplicated rows

# Showing identical rows side by side(
print(all_duplicates.sort_values(by = list(data.columns)).head(10))

# Drop duplicates
data.drop_duplicates(inplace = True)
print('Total Number of duplicated rows after dropping:',data.duplicated().sum())

Total Number of duplicated rows: 157
            State     Sex GeneralHealth  PhysicalHealthDays  MentalHealthDays  \
4712       Alaska    Male     Very good                 0.0               0.0   
7310       Alaska    Male     Very good                 0.0               0.0   
10696     Arizona  Female     Excellent                 0.0               0.0   
11503     Arizona  Female     Excellent                 0.0               0.0   
32137  California  Female           NaN                 NaN               NaN   
32711  California  Female           NaN                 NaN               NaN   
34746  California    Male     Excellent                 0.0               0.0   
35644  California    Male     Excellent                 0.0               0.0   
26789  California    Male     Excellent                 0.0               0.0   
28206  California    Male     Excellent                 0.0               0.0   

                                         LastCheckupTime PhysicalActivi

In [5]:
# Identify Column Data Types
numerical_cols = data.select_dtypes(include = [np.number]).columns.tolist()
categorical_cols = data.select_dtypes(include = [str]).columns.tolist()
print('Numerical Columns:',numerical_cols)
print('Categorical Columns:',categorical_cols)

Numerical Columns: ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']
Categorical Columns: ['State', 'Sex', 'GeneralHealth', 'LastCheckupTime', 'PhysicalActivities', 'RemovedTeeth', 'HadHeartAttack', 'HadAngina', 'HadStroke', 'HadAsthma', 'HadSkinCancer', 'HadCOPD', 'HadDepressiveDisorder', 'HadKidneyDisease', 'HadArthritis', 'HadDiabetes', 'DeafOrHardOfHearing', 'BlindOrVisionDifficulty', 'DifficultyConcentrating', 'DifficultyWalking', 'DifficultyDressingBathing', 'DifficultyErrands', 'SmokerStatus', 'ECigaretteUsage', 'ChestScan', 'RaceEthnicityCategory', 'AgeCategory', 'AlcoholDrinkers', 'HIVTesting', 'FluVaxLast12', 'PneumoVaxEver', 'TetanusLast10Tdap', 'HighRiskLastYear', 'CovidPos']


In [6]:
# Count of unique values in each column
print('Unique values in numerical columns:')
print(data[numerical_cols].nunique())
print('Unique values in categorical columns:')
print(data[categorical_cols].nunique())

Unique values in numerical columns:
PhysicalHealthDays      31
MentalHealthDays        31
SleepHours              24
HeightInMeters         109
WeightInKilograms      599
BMI                   3985
dtype: int64
Unique values in categorical columns:
State                        54
Sex                           2
GeneralHealth                 5
LastCheckupTime               4
PhysicalActivities            2
RemovedTeeth                  4
HadHeartAttack                2
HadAngina                     2
HadStroke                     2
HadAsthma                     2
HadSkinCancer                 2
HadCOPD                       2
HadDepressiveDisorder         2
HadKidneyDisease              2
HadArthritis                  2
HadDiabetes                   4
DeafOrHardOfHearing           2
BlindOrVisionDifficulty       2
DifficultyConcentrating       2
DifficultyWalking             2
DifficultyDressingBathing     2
DifficultyErrands             2
SmokerStatus                  4
ECigaretteUsage

In [7]:
# Dropping the unnecessary columns that are not relevant in predicting risk of heart disease
cols_to_drop = [
    'State', 'TetanusLast10Tdap', 'PneumoVaxEver', 
    'LastCheckupTime', 'ChestScan', 'HIVTesting', 
    'HighRiskLastYear', 'FluVaxLast12'
]
data.drop(columns = cols_to_drop,inplace = True)
print('Total columns after dropping some cols:',data.shape[1])

Total columns after dropping some cols: 32


Feature Selection: Why I Dropped 8 Columns?
I reduced the dataset from 40 to 32 features to ensure a clean and high-performance model
ChestScan — Target leakage: scans ordered because of existing symptoms, inflates accuracy artificially
State — 54 categories = unnecessary dimensionality; lifestyle factors (BMI, smoking) predict better than geography
PneumoVaxEver — >20% missing data; vaccination status = healthcare access proxy, not cardiac risk
TetanusLast10Tdap — >20% missing; unrelated to coronary heart disease biology
HIVTesting — No physiological link to heart disease; measures healthcare-seeking behavior only
FluVaxLast12 — Vaccination records ≠ cardiac protection; adds noise
HighRiskLastYear — Self-reported risk assessment = too subjective; actual biomarkers (BMI, smoking) are stronger
LastCheckupTime — Healthcare access proxy, not disease risk; reflects insurance/income, not physiology

In [8]:
# Check for missing values 
print('Total null values:',data.isnull().sum().sum())

# Drop rows with missing values 
data.dropna(inplace = True)
print('Null values after dropping rows with missing values:',data.isnull().sum().sum())

Total null values: 512890
Null values after dropping rows with missing values: 0


The dataset has 4,45,132 rows and missing values are 5,12,890 which is  5 Lakhs, 12 Thousand, 8 Hundred and Ninety. The reason for greater missing values than  the actual rows can be the multiple missing values in a row.

In [9]:
# Check for whether there is enough data to train the model 
print(f"Original rows: 445132")
print(f"Rows remaining after removing missing values: {len(data)}")
print(f"Total rows lost: {445132 - len(data)}")

Original rows: 445132
Rows remaining after removing missing values: 313274
Total rows lost: 131858


In [10]:
# Outlier detection using IQR method
for col in numerical_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    print(f"Column: {col}, Outliers: {(len(outliers))}")






Column: PhysicalHealthDays, Outliers: 48876
Column: MentalHealthDays, Outliers: 41830
Column: SleepHours, Outliers: 4323
Column: HeightInMeters, Outliers: 1054
Column: WeightInKilograms, Outliers: 7466
Column: BMI, Outliers: 9719


Instead of dropping, we use Domain Knowledge (Medical Cutoffs). We only drop rows that are impossible or data entry errors.
In healthcare dataset we do not drop any rows when there are missing values or outliers but we think according to the medical domain and take decision to either drop or keep the rows.Data is critical in medical domain so, normal data cleaning wont work we need to think accordingly to the domain.

Physical / Mental Health Days: The valid range is 0 to 30.

Sleep Hours: While 1 hour is rare, some people with extreme insomnia or medical conditions actually sleep that little. We usually drop 0 or > 24.

BMI: A BMI up to 80 or 90 is extreme obesity, but real. Drop anything > 100 (likely a typo).

Height / Weight: Drop only if Height is < 0.5 meters (impossible for adults) or Weight is > 300 kg (highly unlikely and usually an error).

In [14]:
# check for current row count 
print(f"Current row count: {len(data)}")
# Use pandas filtering to fetch the rows that do not meet the requirements of an outlier and assign it back to the original DataFrame
data = data[
    (data['PhysicalHealthDays'] >=0) & (data['PhysicalHealthDays'] <= 30) &
    (data['MentalHealthDays'] >=0) & (data['MentalHealthDays'] <= 30) &
    (data['SleepHours'] >=0) & (data['SleepHours'] <= 24) & 
    (data['BMI'] >=10) & (data['BMI'] <=100) & 
    (data['HeightInMeters'] >=0.9) & (data['HeightInMeters'] <=2.5) & 
    (data['WeightInKilograms'] >=30) & (data['WeightInKilograms'] <=300)

]
# Check the row count after removing outliers
print(f"Row count after removing outliers: {len(data)}")



Current row count: 313274
Row count after removing outliers: 313269


"After applying clinical cutoffs, only 5 records were identified as biologically impossible (e.g., SleepHours > 24). While statistical methods like IQR flagged 110,000+ entries as outliers, domain analysis confirmed these represent high-risk individuals (e.g., chronic illness or obesity) rather than data errors. Retaining 99.99% of these 'outliers' ensures the model captures critical health patterns."